# Faruq-v3 — MHC0 vs MHF1 one-seed screening

Melatih dua capacity-matched head pada seed 42: P5 control dan P3+P4+P5 fusion. D0 dipakai ulang. Test tetap tidak tersedia. Checkpoint ditulis langsung ke shared Drive dan otomatis resume setelah runtime/account berganti.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_summary.json',
    'experiments/faruq-v3-multilevel-head-v1/static_audit.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
BASELINE = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_summary.json')
STATIC_AUDIT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-multilevel-head-v1/static_audit.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-multilevel-head-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file()
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT :', PROJECT_ROOT)
print('DATA    :', DATA_ROOT)
print('OUTPUT  :', OUTPUT_ROOT)
for code in ('MHC0', 'MHF1'):
    last = OUTPUT_ROOT / f'{code}_seed42/weights/last.pt'
    best = OUTPUT_ROOT / f'{code}_seed42/weights/best.pt'
    print(code, 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START'))

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_multilevel_head',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--baseline-summary', str(BASELINE),
    '--static-audit', str(STATIC_AUDIT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY = OUTPUT_ROOT / 'val_reports/multilevel_head_seed42_decision.json'
assert SUMMARY.is_file(), f'Screening belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = [{'model': name, **metrics} for name, metrics in result['results'].items()]
display(pd.DataFrame(rows).style.format({name: '{:.2%}' for name in ('macro_map50_95', 'bottom3_class_map50_95', 'worst_class_map50_95')}))
print('COMPARISONS:', json.dumps(result['comparisons'], indent=2))
print('CRITERIA   :', result['criteria'])
print('DECISION   :', result['decision'])
print('NEXT       :', result['next_action'])
print('SUMMARY    :', SUMMARY)
print('Kirim tabel dan keputusan. Jangan membuka test atau menjalankan seed lain.')